## Project FAKE V/S REAL NEWS PREDICTION

#### Import important library

In [42]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Importing datasets

In [43]:
fake = pd.read_csv('Fake.csv')
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [44]:
fake.shape

(23481, 4)

In [45]:
true = pd.read_csv('True.csv')

In [46]:
true.shape

(21417, 4)

In [47]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


### Checking out the data

In [48]:
true['subject'].value_counts()

subject
politicsNews    11272
worldnews       10145
Name: count, dtype: int64

### Labelled the real as 1 and fake as 0

In [49]:
fake['label'] = 0
true['label'] = 1

### Concatenated the two csv files to make a dataset for feeding to the model

In [50]:
news = pd.concat([fake, true], ignore_index = True)

### Shuffled the rows to make sure it is random

In [51]:
news = news.sample(frac=1, random_state=42).reset_index(drop=True)

In [52]:
news.shape

(44898, 5)

### Drop the Subject and Date column
It is because model might learn all the true news through its source and date. This may cause overfitting.

In [53]:
news=news.drop(['subject', 'date'], axis=1)

### Simple data cleaning

In [54]:
news.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   title   44898 non-null  object
 1   text    44898 non-null  object
 2   label   44898 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.0+ MB


In [55]:
news.duplicated().sum()

np.int64(5793)

In [56]:
news.drop_duplicates(inplace=True)

In [57]:
news.shape

(39105, 3)

In [58]:
news.iloc[10:20,:]

,title,text,label
10,New Jersey's Christie mulls run to lead Republ...,WASHINGTON (Reuters) - New Jersey Governor Chr...,1
11,WHERE’S HILLARY? CLINTON SPOTTED Dining Alone,,0
12,"France, Germany want Iran to reverse ballistic...",PARIS (Reuters) - France and Germany agree tha...,1
13,Aide to EU Commission head tweets picture of w...,"BRUSSELS (Reuters) - Martin Selmayr, a top aid...",1
14,"Trump Issues Warning To “200,000 man army” Tha...",Before you blame Trump for not being compassio...,0
15,U.S. gives Laos extra $90 million to help clea...,VIENTIANE (Reuters) - The United States announ...,1
16,JUDGE DECLARES BABY NAME “Illegal” To Prevent ...,Does a judge have the right to determine what ...,0
17,Paul Ryan Takes A Monumentally HUMILIATING Ph...,Politicians are all about photo-ops. This is e...,0
18,"Republicans Dine With Trump, Then Try To Rail...",Republicans are once again putting their party...,0
19,U.S. House panel offers alternative to retirem...,WASHINGTON (Reuters) - Legislation to bypass t...,1


In [59]:
news['text'].isnull().sum()

np.int64(0)

In [60]:
news[news['text']==' '].count()

title    445
text     445
label    445
dtype: int64

In [61]:
news=news[news['text']!=" "]

### More cleaning
Most of the real news has "LOCATION (Reuters) -" format and this must be removed to avoid overfitting.

In [62]:
news['text'].str.contains(r'\(Reuters\)', regex=True).sum()

np.int64(21035)

In [63]:
news[news['label'] == 1].count()

title    21196
text     21196
label    21196
dtype: int64

In [64]:
import re
def strip_reuters(text):
    text=re.sub(r'^.*?\(Reuters\)\s*-', '', text)
    return text

In [65]:
before = news['text'].str.contains(r'\(Reuters\)', regex=True).sum()

In [66]:
news['text'] = news['text'].apply(strip_reuters)

In [67]:
after = news['text'].str.contains(r'\(Reuters\)', regex=True).sum()

10 rows aren't that important and maybe left as residuals.

In [68]:
print('after =',after,'\n','before = ', before)

after = 10 
 before =  21035


In [69]:
news

,title,text,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",0
1,Trump drops Steve Bannon from National Securit...,U.S. President Donald Trump removed his chief...,1
2,Puerto Rico expects U.S. to lift Jones Act shi...,Puerto Rico Governor Ricardo Rossello said on...,1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",0
4,Donald Trump heads for Scotland to reopen a go...,Most U.S. presidential candidates go abroad t...,1
...,...,...,...
44890,Burundi opposition platform boycotts new round...,Burundi s main opposition grouping said it is...,1
44892,Hillary’s Message To Former Miss Universe Cal...,Miss Universe 1996 Alicia Machado is now an Am...,0
44894,PM May seeks to ease Japan's Brexit fears duri...,British Prime Minister Theresa May will look ...,1
44895,Merkel: Difficult German coalition talks can r...,Chancellor Angela Merkel said German parties ...,1


In [70]:
news[news['text'].str.contains(r'\(Reuters\)', regex=True)]

,title,text,label
5687,"Trump to meet Australian PM, relations straine...",SYDNEY/WASHINGTON (Reuters)) - U.S. President ...,1
7399,BRILLIANT NEWT GINGRICH: Here’s Why Trump Shou...,Wouldn t you know that the CIA director had wo...,0
21481,VOTING MACHINES STOLEN In Controversial GA Ele...,"SANDY SPRINGS, Ga., April 18 (Reuters) For U...",0
22434,The Trump presidency on Jan 29 at 4:12 P.M. ES...,Jan 29 (Reuters) - Highlights of the day for ...,1
26634,BREAKING: OBAMA’S DOJ TO FAST TRACK $29 MILLIO...,This government grant is taxpayer dollars and ...,0
27215,Exclusive: Trump son-in-law had undisclosed co...,U.S. President Donald Trump’s son-in-law and ...,1
27448,INVASION OF AMERICA: POSSIBILITY OF A TRUMP WA...,A report just came out about the crazy huge nu...,0
27800,Why John McCain Will Never Vote to Repeal and ...,It s all about the money isn t that the way it...,0
36425,BREAKING: RUSSIA SMELLS OBAMA’S FOREIGN POLICY...,Leading from behind hasn t worked out so well ...,0
37685,Democrats Go On The Attack Against Voter ID Laws,Virginia s Voter ID law is being challenged in...,0


### Check the real vs fake text's length